# 🧠 VLM Facial Authentication — Advanced Demo

This notebook demonstrates Vision Language Model (VLM) reasoning for facial authentication.

## Approaches Covered
1. **Approach 1 — VLM Judge**: VLM reviews registration vs auth frames after traditional pipeline grants
2. **Approach 2 — Dual-Video VLM**: Full video comparison with fused reasoning
3. **Approach 3 — VLM Brain**: VLM as the primary decision-maker

## Models
- **Qwen2.5-VL-3B-Instruct** (4-bit quantized) — primary, needs T4 GPU
- **moondream2 (1.9B)** — fallback, works on CPU

## Requirements
- Google Colab (Free Tier with T4 GPU recommended)
- ~3GB VRAM for Qwen2.5-VL (4-bit)
- ~12GB system RAM

## 1. Setup & Installation

In [ ]:
# Install required packages
!pip install -q transformers>=4.45.0 accelerate>=0.25.0 bitsandbytes>=0.41.0
!pip install -q qwen-vl-utils>=0.0.2 sentencepiece protobuf einops
!pip install -q opencv-python-headless Pillow
!pip install -q gradio>=4.0.0
!pip install -q timm>=0.9.0

print("✅ All packages installed")

In [ ]:
# Detect hardware
import torch
import psutil

print("=" * 50)
print("HARDWARE DETECTION")
print("=" * 50)
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"VRAM: {vram:.1f} GB")
else:
    print("GPU: None (CPU mode)")

ram = psutil.virtual_memory()
print(f"System RAM: {ram.total / (1024**3):.1f} GB")
print(f"Available RAM: {ram.available / (1024**3):.1f} GB")
print("=" * 50)

USE_GPU = torch.cuda.is_available()
DEVICE = "cuda" if USE_GPU else "cpu"
print(f"\n🎯 Using: {DEVICE.upper()}")

## 2. Load VLM Model

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info
from PIL import Image
import time

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

print(f"Loading {MODEL_ID} with 4-bit quantization...")
t0 = time.time()

if USE_GPU:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quant_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.float16,
    )
else:
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.float32,
    )

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

load_time = time.time() - t0
print(f"✅ Model loaded in {load_time:.1f}s")

if USE_GPU:
    allocated = torch.cuda.memory_allocated() / (1024**3)
    print(f"GPU Memory used: {allocated:.2f} GB")

In [ ]:
import json
import re
import numpy as np
import cv2

def vlm_inference(images, prompt, max_tokens=512, temperature=0.1):
    """Run VLM inference with one or more images."""
    content = []
    for img in images:
        if isinstance(img, np.ndarray):
            img = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        content.append({"type": "image", "image": img})
    content.append({"type": "text", "text": prompt})

    messages = [{"role": "user", "content": content}]
    text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text_input], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt"
    ).to(model.device)

    t0 = time.time()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=max_tokens,
            temperature=temperature, do_sample=temperature > 0
        )
    inference_time = (time.time() - t0) * 1000

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    output_text = processor.decode(generated_ids, skip_special_tokens=True)

    return output_text, inference_time

def parse_json_output(text):
    """Parse JSON from VLM output."""
    text = text.strip()
    try:
        return json.loads(text)
    except:
        pass
    match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    if match:
        try: return json.loads(match.group(1))
        except: pass
    match = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', text, re.DOTALL)
    if match:
        try: return json.loads(match.group(0))
        except: pass
    return {"reasoning": text, "overall_score": 0.5}

print("✅ Inference functions ready")

## 3. Upload Test Images

Upload registration and authentication face images to test.

In [ ]:
from google.colab import files
import io

print("📸 Upload REGISTRATION image(s):")
reg_uploads = files.upload()
reg_images = []
for name, data in reg_uploads.items():
    img = Image.open(io.BytesIO(data)).convert("RGB")
    img = img.resize((512, 512))
    reg_images.append(img)
    print(f"  ✅ {name}: {img.size}")

print(f"\n📸 Upload AUTHENTICATION image(s):")
auth_uploads = files.upload()
auth_images = []
for name, data in auth_uploads.items():
    img = Image.open(io.BytesIO(data)).convert("RGB")
    img = img.resize((512, 512))
    auth_images.append(img)
    print(f"  ✅ {name}: {img.size}")

print(f"\n✅ Loaded {len(reg_images)} registration + {len(auth_images)} authentication images")

## 4. Approach 1 — VLM Judge

VLM acts as a judge reviewing registration vs authentication frames.

In [ ]:
JUDGE_PROMPT = """You are a facial authentication security system. Your job is to compare a registered user's face with an authentication attempt.

TASK: Analyze the provided images carefully.
- The FIRST image(s) are from the user's REGISTRATION (reference identity).
- The LAST image(s) are from the current AUTHENTICATION attempt.

Evaluate:
1. IDENTITY: Are they the same person? (bone structure, nose, eyes, jawline)
2. LIVENESS: Is the auth image a live person? (skin texture, reflections, depth)
3. AUTHENTICITY: Any spoofing/deepfake signs? (blending, unnatural smoothness)

Respond ONLY in JSON:
{
  "same_person": true/false,
  "same_person_confidence": 0.0-1.0,
  "is_live": true/false,
  "liveness_confidence": 0.0-1.0,
  "is_authentic": true/false,
  "authenticity_confidence": 0.0-1.0,
  "overall_score": 0.0-1.0,
  "reasoning": "detailed explanation",
  "red_flags": []
}"""

n_reg = len(reg_images)
n_auth = len(auth_images)
context = f"\n\nNote: {n_reg + n_auth} images total. First {n_reg} = REGISTRATION, last {n_auth} = AUTHENTICATION."

print("🧠 Running Approach 1: VLM Judge...")
all_images = reg_images + auth_images
output, time_ms = vlm_inference(all_images, JUDGE_PROMPT + context)

print(f"\n⏱️ Inference time: {time_ms:.0f}ms")
print(f"\n📝 Raw output:\n{output}")

result = parse_json_output(output)
print(f"\n📊 Parsed result:")
for key, val in result.items():
    if key != 'reasoning':
        print(f"  {key}: {val}")
print(f"\n💬 Reasoning: {result.get('reasoning', 'N/A')}")

## 5. Approach 2 — Dual-Video VLM Reasoning

Enhanced comparison with detailed multi-aspect analysis.

In [ ]:
DUAL_VIDEO_PROMPT = """You are an advanced facial authentication AI performing a comprehensive security analysis.

You are comparing REGISTRATION video frames with AUTHENTICATION video frames.

REGISTRATION frames (reference identity): Images 1-{n_reg}
AUTHENTICATION frames (verification attempt): Images {n_reg+1}-{total}

Perform a DETAILED multi-aspect analysis:

## IDENTITY VERIFICATION
- Facial geometry: Compare interpupillary distance, nose-to-lip ratio, jawline angle
- Unique features: Moles, scars, birthmarks, facial hair patterns
- Aging consistency: Does the age appear consistent between registration and auth?

## LIVENESS DETECTION
- Skin analysis: Real skin has pores, fine lines, subtle color variations
- Eye analysis: Real eyes have complex reflections, visible veins, moisture
- Expression naturalness: Are micro-expressions present? Do expressions look organic?
- Environmental cues: Is the lighting from a real environment?

## ANTI-SPOOFING
- Screen attack: Look for moire patterns, screen bezels, pixel grid artifacts
- Print attack: Look for paper texture, flat lighting, ink artifacts
- Mask attack: Look for mask edges, unnatural skin boundaries
- Deepfake: Look for blending artifacts, unnatural skin smoothness, warping

## TEMPORAL ANALYSIS (between frames)
- Natural movement: Is there natural head movement between frames?
- Consistency: Are lighting and features consistent across frames?
- Any static/frozen appearance suggesting a static image?

Provide your analysis as JSON:
{{
  "identity_match": {{"verdict": true/false, "confidence": 0.0-1.0, "details": "..."}},
  "liveness": {{"verdict": true/false, "confidence": 0.0-1.0, "details": "..."}},
  "anti_spoof": {{"verdict": true/false, "confidence": 0.0-1.0, "details": "..."}},
  "temporal": {{"verdict": true/false, "confidence": 0.0-1.0, "details": "..."}},
  "overall_decision": "GRANT" or "DENY",
  "overall_confidence": 0.0-1.0,
  "reasoning_summary": "comprehensive summary",
  "red_flags": []
}}""".format(n_reg=len(reg_images), total=len(reg_images)+len(auth_images))

print("🧠 Running Approach 2: Dual-Video VLM Reasoning...")
all_images_2 = reg_images + auth_images
output2, time_ms2 = vlm_inference(all_images_2, DUAL_VIDEO_PROMPT)

print(f"\n⏱️ Inference time: {time_ms2:.0f}ms")
print(f"\n📝 Raw output:\n{output2}")

result2 = parse_json_output(output2)
print(f"\n📊 Detailed Analysis:")
for key in ['identity_match', 'liveness', 'anti_spoof', 'temporal']:
    if key in result2:
        v = result2[key]
        print(f"  {key}: verdict={v.get('verdict')}, confidence={v.get('confidence')}")
        print(f"    → {v.get('details', 'N/A')}")

print(f"\n🎯 Decision: {result2.get('overall_decision', 'N/A')}")
print(f"   Confidence: {result2.get('overall_confidence', 'N/A')}")
print(f"\n💬 Summary: {result2.get('reasoning_summary', 'N/A')}")

## 6. Approach 3 — VLM Brain

VLM as the primary decision-maker with detailed face description.

In [ ]:
# Step 1: Generate face description from registration
DESCRIBE_PROMPT = """Analyze this face image in detail for identity verification purposes.

Create a detailed biometric description including:
1. Face shape and proportions
2. Eye shape, color, spacing
3. Nose shape and size
4. Lip shape and proportions
5. Jawline and chin
6. Eyebrow shape
7. Skin characteristics
8. Any unique identifying features (moles, scars, etc.)
9. Estimated age range
10. Overall distinguishing characteristics

Provide as JSON:
{
  "face_description": {
    "face_shape": "...",
    "eyes": "...",
    "nose": "...",
    "lips": "...",
    "jawline": "...",
    "eyebrows": "...",
    "skin": "...",
    "unique_features": "...",
    "age_range": "...",
    "distinguishing_chars": "..."
  },
  "identity_hash": "a short 2-3 sentence summary that uniquely identifies this person"
}"""

print("🧠 Approach 3 — Step 1: Generating face description from registration...")
desc_output, desc_time = vlm_inference(reg_images[:1], DESCRIBE_PROMPT)
print(f"⏱️ Description time: {desc_time:.0f}ms")
reg_description = parse_json_output(desc_output)
print(f"\n📋 Registration face description:")
if 'face_description' in reg_description:
    for key, val in reg_description['face_description'].items():
        print(f"  {key}: {val}")
print(f"\n🆔 Identity hash: {reg_description.get('identity_hash', 'N/A')}")

In [ ]:
# Step 2: Verify authentication against description
identity_hash = reg_description.get('identity_hash', 'No description available')

VERIFY_PROMPT = f"""You are the primary decision-maker for facial authentication.

REGISTERED IDENTITY DESCRIPTION:
{identity_hash}

TASK: Look at the authentication image and determine if this person matches the description above.
Also assess if this is a live, real person (not a photo, screen, mask, or deepfake).

Respond in JSON:
{{
  "matches_description": true/false,
  "match_confidence": 0.0-1.0,
  "is_live_person": true/false,
  "liveness_confidence": 0.0-1.0,
  "decision": "GRANT" or "DENY",
  "reasoning": "detailed explanation comparing against the stored description",
  "discrepancies": ["list any differences from the description"]
}}"""

print("🧠 Approach 3 — Step 2: Verifying authentication against description...")
verify_output, verify_time = vlm_inference(auth_images[:1], VERIFY_PROMPT)
print(f"⏱️ Verification time: {verify_time:.0f}ms")

verify_result = parse_json_output(verify_output)
print(f"\n🎯 VLM Brain Decision: {verify_result.get('decision', 'N/A')}")
print(f"   Match: {verify_result.get('matches_description', 'N/A')} ({verify_result.get('match_confidence', 'N/A')})")
print(f"   Live: {verify_result.get('is_live_person', 'N/A')} ({verify_result.get('liveness_confidence', 'N/A')})")
print(f"\n💬 Reasoning: {verify_result.get('reasoning', 'N/A')}")
if verify_result.get('discrepancies'):
    print(f"\n⚠️ Discrepancies: {verify_result['discrepancies']}")

## 7. Comparison Summary

In [ ]:
print("=" * 70)
print("APPROACH COMPARISON SUMMARY")
print("=" * 70)
print(f"")
print(f"{'Approach':<25} {'Decision':<12} {'Confidence':<12} {'Time (ms)':<12}")
print(f"{'-' * 25} {'-' * 12} {'-' * 12} {'-' * 12}")

# Approach 1
a1_score = result.get('overall_score', 0.5)
a1_decision = 'GRANT' if a1_score >= 0.55 else 'DENY'
print(f"{'1. VLM Judge':<25} {a1_decision:<12} {a1_score:<12.3f} {time_ms:<12.0f}")

# Approach 2
a2_conf = result2.get('overall_confidence', 0.5)
a2_decision = result2.get('overall_decision', 'N/A')
print(f"{'2. Dual-Video VLM':<25} {a2_decision:<12} {a2_conf:<12.3f} {time_ms2:<12.0f}")

# Approach 3
a3_conf = verify_result.get('match_confidence', 0.5)
a3_decision = verify_result.get('decision', 'N/A')
a3_time = desc_time + verify_time
print(f"{'3. VLM Brain':<25} {a3_decision:<12} {a3_conf:<12.3f} {a3_time:<12.0f}")

print(f"")
print(f"GPU Memory: {torch.cuda.memory_allocated()/(1024**3):.2f} GB" if USE_GPU else "Mode: CPU")
print("=" * 70)

## 8. Interactive Demo with Gradio

In [ ]:
import gradio as gr

def vlm_authenticate(reg_image, auth_image, approach):
    """Run VLM authentication."""
    if reg_image is None or auth_image is None:
        return "Please upload both images.", {}

    reg_pil = Image.fromarray(reg_image).resize((512, 512))
    auth_pil = Image.fromarray(auth_image).resize((512, 512))

    if approach == "Approach 1: VLM Judge":
        prompt = JUDGE_PROMPT + "\n\nNote: 2 images. First = REGISTRATION, Second = AUTHENTICATION."
    elif approach == "Approach 2: Dual-Video":
        prompt = DUAL_VIDEO_PROMPT
    else:
        prompt = VERIFY_PROMPT

    output, time_ms = vlm_inference([reg_pil, auth_pil], prompt)
    parsed = parse_json_output(output)

    reasoning = parsed.get('reasoning', parsed.get('reasoning_summary', output[:500]))
    score = parsed.get('overall_score', parsed.get('overall_confidence', 0.5))

    summary = f"""## Result
- **Decision**: {'✅ GRANT' if score >= 0.55 else '❌ DENY'}
- **Confidence**: {score:.1%}
- **Inference Time**: {time_ms:.0f}ms

## Reasoning
{reasoning}
"""
    return summary, parsed

demo = gr.Interface(
    fn=vlm_authenticate,
    inputs=[
        gr.Image(label="Registration Image", type="numpy"),
        gr.Image(label="Authentication Image", type="numpy"),
        gr.Radio(
            ["Approach 1: VLM Judge", "Approach 2: Dual-Video", "Approach 3: VLM Brain"],
            value="Approach 1: VLM Judge",
            label="Select Approach"
        )
    ],
    outputs=[
        gr.Markdown(label="Analysis Result"),
        gr.JSON(label="Raw JSON Output")
    ],
    title="🧠 VLM Facial Authentication Demo",
    description="Upload registration and authentication face images to test VLM reasoning.",
    examples=[],
)

demo.launch(share=True, debug=True)